Apache Spark is a **unified, open-source distributed computing engine** for large-scale data processing. It lets you run computations across a cluster of machines in parallel — handling workloads that range from batch ETL, to SQL queries, to machine learning, to real-time streaming — all within a single framework.

In this notebook we'll cover:
- How and why Spark was created
- What makes it fast
- Its core components and ecosystem
- Where Spark fits compared to alternatives

## A Brief History

| Year | Milestone |
|------|-----------|
| **2009** | Spark started as a research project at UC Berkeley's AMPLab by Matei Zaharia |
| **2010** | First paper published: *Spark: Cluster Computing with Working Sets* |
| **2012** | RDD paper published — formal foundation of Spark's data model |
| **2013** | Donated to the Apache Software Foundation; Databricks founded |
| **2014** | Apache Spark 1.0 released; became a top-level Apache project |
| **2015** | Spark 1.3 introduced DataFrames; Spark beat Hadoop MapReduce in sorting benchmarks |
| **2016** | Spark 2.0 — unified DataFrame/Dataset API, Structured Streaming, SparkSession |
| **2020** | Spark 3.0 — Adaptive Query Execution (AQE), better Python/pandas support |
| **2022** | Spark 3.3 — Pandas API on Spark (`pyspark.pandas`) made stable |
| **2023** | Spark 3.4/3.5 — Spark Connect (remote client), improved Python UDF performance |

### The Problem Spark Was Solving

Before Spark, the dominant framework for distributed processing was **Hadoop MapReduce**. It worked, but it had a painful limitation: every intermediate step had to be written to disk (HDFS). A multi-step ML algorithm might do 100 iterations — each one triggering a full read/write cycle to disk. That made iterative algorithms extremely slow.

Spark's key insight was simple: **keep data in memory across iterations**. This made iterative workloads (machine learning, graph algorithms) 10–100× faster than MapReduce.

## Why Spark is Fast

Spark's speed comes from three core design decisions:

### 1. In-Memory Processing
Intermediate results are kept in RAM across stages instead of being written to disk between each step. Disk I/O is orders of magnitude slower than memory access — this alone is the biggest win for iterative jobs.

### 2. Lazy Evaluation
Spark does **not** execute transformations immediately. When you call `.filter()` or `.groupBy()`, Spark records the instruction but does nothing. Execution is triggered only by an **action** (e.g., `.count()`, `.show()`, `.write()`). This lets Spark build a complete picture of your job — the **DAG (Directed Acyclic Graph)** — before running anything.

### 3. Catalyst Optimizer
Before executing a query, Spark's **Catalyst** optimizer rewrites it into a more efficient plan — pushing filters early, reordering joins, eliminating redundant steps. You write simple code; Spark figures out the fastest execution path.

![Catalyst Optimizer Pipeline](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/catalyst-optimizer.svg)

## Core Components

Spark is a **unified** platform — one engine, multiple workload types:

![Spark Components Architecture](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-components.svg)

### Spark Core
The foundation. Handles task scheduling, memory management, fault recovery, and I/O. Provides the **RDD (Resilient Distributed Dataset)** API — the original low-level abstraction.

### Spark SQL & DataFrames
The most widely used component. Lets you query structured data with SQL or a DataFrame API (similar to pandas). Built on top of the Catalyst optimizer and Tungsten execution engine. Supports reading/writing Parquet, ORC, JSON, CSV, Delta Lake, and more.

### Structured Streaming
Real-time data processing built on the DataFrame API. You write the same batch code — Spark handles micro-batching or continuous processing under the hood. Supports exactly-once semantics and stateful operations.

### MLlib
Spark's distributed machine learning library. Provides ML algorithms (classification, regression, clustering, recommendation), feature engineering tools, and `ML Pipelines` — a scikit-learn-inspired API for building reproducible ML workflows.

### GraphX
A graph computation library for building and transforming graphs at scale (e.g., PageRank, connected components). Primarily Scala/Java; Python users typically use GraphFrames instead.

## Spark vs. Hadoop MapReduce

| | Hadoop MapReduce | Apache Spark |
|---|---|---|
| **Processing model** | Disk-based (read → process → write) | In-memory (keeps data in RAM) |
| **Speed** | Slow for iterative jobs | 10–100× faster for iterative jobs |
| **APIs** | Java only (low-level) | Python, Scala, Java, R (high-level) |
| **Workloads** | Batch only | Batch, SQL, Streaming, ML, Graph |
| **Fault tolerance** | Replicates data to HDFS | Re-computes lost partitions via lineage |
| **Ease of use** | Verbose, boilerplate-heavy | Concise, expressive APIs |

> Spark does **not** replace HDFS — it often reads from and writes to HDFS. What it replaces is the MapReduce *execution engine*.

## Key Features at a Glance

- **Speed** — In-memory computation + Catalyst optimizer
- **Ease of use** — High-level APIs in Python, Scala, Java, R; interactive shell
- **Unified engine** — One tool for batch, SQL, streaming, ML, graph
- **Fault tolerance** — RDD lineage; lost data is recomputed, not re-fetched
- **Scalability** — Runs on a laptop (local mode) to thousands of nodes
- **Open ecosystem** — Reads from S3, HDFS, GCS, Kafka, JDBC, Delta Lake, Iceberg, and more
- **Managed platforms** — Databricks, AWS EMR, Google Dataproc, Azure HDInsight, Azure Databricks

## Common Use Cases

| Use Case | Example |
|---|---|
| **ETL / Data Engineering** | Ingest raw logs → clean → write to data warehouse |
| **Ad-hoc SQL analytics** | Query billions of rows of Parquet files with Spark SQL |
| **Machine Learning** | Train a model on a 100 GB dataset distributed across a cluster |
| **Real-time streaming** | Process Kafka events to detect fraud in milliseconds |
| **Data lake processing** | Read/write Delta Lake tables with ACID guarantees |
| **Graph analytics** | Run PageRank on a social graph with billions of edges |

## Hands-On: Your First Spark Job

Let's start a SparkSession and run a minimal job to see Spark in action.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("WhatIsSpark") \
    .master("local[*]") \
    .getOrCreate()

spark.version

In [ ]:
# Create a small DataFrame from a Python list
data = [
    ("Apache Spark",  2009, "AMPLab / UC Berkeley"),
    ("Hadoop",        2006, "Yahoo"),
    ("Flink",         2011, "TU Berlin"),
    ("Kafka",         2011, "LinkedIn"),
    ("Databricks",    2013, "Spark founders"),
]
columns = ["project", "year_born", "origin"]

df = spark.createDataFrame(data, columns)
df.show()

In [ ]:
# Filter + select — these are lazy transformations
result = df.filter(df.year_born >= 2011).select("project", "year_born")

# .show() is an action — this triggers execution
result.show()

In [ ]:
# Inspect the execution plan Spark built
# 'Project' = column selection, 'Filter' = the .filter() call
result.explain()

In [ ]:
# Run a SQL query on the same data
df.createOrReplaceTempView("big_data_projects")

spark.sql("""
    SELECT origin, COUNT(*) AS projects
    FROM big_data_projects
    GROUP BY origin
    ORDER BY projects DESC
""").show()

## Summary

- Spark was born in 2009 at UC Berkeley to solve the speed limitations of Hadoop MapReduce, primarily by keeping data **in memory**.
- It is a **unified engine**: one framework covers batch, SQL, streaming, ML, and graph workloads.
- Speed comes from **in-memory processing**, **lazy evaluation**, and the **Catalyst optimizer**.
- The five main components are: **Spark Core**, **Spark SQL**, **Structured Streaming**, **MLlib**, and **GraphX**.
- You interact with Spark primarily through the **DataFrame API** (Python/PySpark) or SQL — both compile down to the same optimized execution engine.